# Spark execution
Today we will dive into the execution internals of spark. We will be reading some (fake) transactions and perform some transformations on them. While doing these transformations, we aim to get more insights into how spark works. We will check wide and narrow transformations, shuffles and spark plans to understand what is happening behind the code.

## First we will import the pyspark library and create a spark session

In [ ]:
from pyspark.sql import SparkSession, functions as sf

In [2]:
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/20 09:17:29 WARN Utils: Your hostname, codespaces-d08206, resolves to a loopback address: 127.0.0.1; using 10.0.0.175 instead (on interface eth0)
26/05/20 09:17:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 09:17:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Now we will read the data and we will show the first records

In [6]:
df = spark.read.csv("file:///workspaces/spark-workshop/transactions.csv", header=True)

Run the cell below, what do you see?

In [7]:
df

DataFrame[Amount: string, Currency: string, Payer account number: string, Beneficiary account number: string, Payer country code: string, Beneficiary country code: string, Transaction timestamp: string]

Now run the show command.

In [8]:
df.show()

+-------+--------+--------------------+--------------------------+------------------+------------------------+---------------------+
| Amount|Currency|Payer account number|Beneficiary account number|Payer country code|Beneficiary country code|Transaction timestamp|
+-------+--------+--------------------+--------------------------+------------------+------------------------+---------------------+
|3383.64|     EUR|  HU28EXDV9600133890|      FR71GWUW379402654...|                HU|                      FR| 2025-07-10T05:06:54Z|
| 769.08|     DKK|  GB93ORDM8495931034|      GB10HDMI525534192...|                GB|                      GB| 2025-02-03T17:58:29Z|
|4627.42|     CAD|IE29BZKM139537672...|      RO33XSNS532871012...|                IE|                      RO| 2025-02-13T10:45:32Z|
|8858.91|     JPY|AU87XDVR514627048...|      SE77GELY880957015430|                AU|                      SE| 2025-01-09T22:01:18Z|
| 224.86|     USD|IE08YRYE782489638346|      PL85ULOQ133150983...|   

Do you recognize what just happened? Remember lazy evaluation and actions?

We called show, an action, and spark started to execute.

## Make a column where we want to check if this is a domestic transaction

In [ ]:
df = df.withColumn(
    "is_domestic",
    ...
)

## We want to only keep eur and usd transactions, create a filter for that

In [ ]:
df_eur_usd = df.where(
    ...
)

## Let's check how spark executes this

In [ ]:
df_eur_usd.explain()

## What do you see?

## What happens when we filter on only domestic transactions?

In [ ]:
df_domestic = df.where(
    ...
)

In [ ]:
df_domestic.explain()

## We want to convert the country codes to full country names

Run the code below to prevent spark from converting your join to broadcast hash joins

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

First, read in the conversion table. Then perform a left join.

In [ ]:
country_df = ...
df_smj = df.join(
    country_df,
    ...
)

In [ ]:
df.explain()

What do you notice?

In [ ]:
df_bhj = df.join(
    sf.broadcast(country_df),
    ...
)

Does this plan look different than before? Which one do you think will be faster?

## Now we would like to know the total amounts transferred for each currency

In [ ]:
total_currency = df.groupBy(
    ...
)

In [ ]:
total_currency.show()

In [ ]:
total_currency.explain()

What do you notice?